# 🌬️ MONITOR DE CALIDAD DEL AIRE - VALLE DE ABURRÁ (SIATA)
## Sistema de Consulta y Análisis del Índice de Calidad del Aire (ICA)

**Proyecto Académico - Componente Práctico**

- **Autor:** [Tu Nombre Aquí]  
- **Fecha:** 2026  
- **Institución:** [Tu Universidad]

### 📑 Contenido del Notebook
1. 🛠️ Configuración del Entorno
2. 🌐 Obtención de Datos desde la API de SIATA
3. 🧹 Limpieza y Transformación de Datos
4. 📊 Visualización y Análisis Exploratorio
5. 🗺️ Georreferenciación de Estaciones
6. 📈 Resumen Estadístico y Conclusiones

## 1. 🛠️ CONFIGURACIÓN DEL ENTORNO

In [ ]:
# Instalación de dependencias (silenciosa)
!pip install requests pandas numpy plotly folium -q
print("✅ Entorno configurado y librerías listas.")

In [ ]:
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import MarkerCluster
import warnings

# Ignorar advertencias menores para una salida más limpia
warnings.filterwarnings('ignore')
print("✅ Librerías importadas exitosamente.")

## 2. 🌐 OBTENCIÓN DE DATOS DESDE LA API DE SIATA

In [ ]:
def obtener_datos_siata(tipo_contaminante='pm25'):
    """
    Realiza la petición a la API pública de SIATA para obtener las últimas mediciones.
    
    Args:
        tipo_contaminante (str): 'pm25', 'pm10' o 'ozono'
    Returns:
        dict: Datos en formato JSON o None si falla la conexión.
    """
    url_base = 'https://siata.gov.co/EntregaData1/'
    endpoints = {
        'pm25': 'Datos_SIATA_Aire_AQ_pm25_Last.json',
        'pm10': 'Datos_SIATA_Aire_AQ_pm10_Last.json',
        'ozono': 'Datos_SIATA_Aire_AQ_o3_Last.json'
    }
    
    url_completa = f"{url_base}{endpoints.get(tipo_contaminante, endpoints['pm25'])}"
    
    try:
        print(f"🔄 Conectando con SIATA ({tipo_contaminante.upper()})...")
        respuesta = requests.get(url_completa, timeout=15)
        respuesta.raise_for_status()
        datos = respuesta.json()
        print(f"✅ ¡Éxito! Se descargaron {len(datos.get('measurements', []))} registros.")
        return datos
    except requests.exceptions.RequestException as error:
        print(f"❌ Fallo en la conexión: {error}")
        return None

# Ejecutar la consulta para PM2.5
datos_crudos = obtener_datos_siata('pm25')

## 3. 🧹 LIMPIEZA Y TRANSFORMACIÓN DE DATOS

In [ ]:
if datos_crudos:
    # Crear DataFrame inicial
    df = pd.DataFrame(datos_crudos['measurements'])
    
    # Desanidar columnas complejas de forma segura con .get()
    df['fecha_utc'] = pd.to_datetime(df['date'].apply(lambda x: x.get('utc')))
    df['fecha_local'] = pd.to_datetime(df['date'].apply(lambda x: x.get('local')))
    df['latitud'] = df['coordinates'].apply(lambda x: x.get('latitude'))
    df['longitud'] = df['coordinates'].apply(lambda x: x.get('longitude'))
    
    # Estandarizar nombres de columnas
    df = df.rename(columns={
        'value': 'valor_pm25',
        'location': 'estacion',
        'city': 'municipio',
        'unit': 'unidad_medida'
    })
    
    # Manejo de valores perdidos específicos de SIATA (-9999)
    df['valor_pm25'] = df['valor_pm25'].replace(-9999, np.nan)
    
    # Filtrar solo datos válidos (sin nulos en columnas críticas)
    df_limpio = df.dropna(subset=['valor_pm25', 'fecha_local', 'latitud', 'longitud']).copy()
    
    print(f"📊 Dimensiones finales: {df_limpio.shape[0]} filas x {df_limpio.shape[1]} columnas")
    display(df_limpio[['fecha_local', 'estacion', 'valor_pm25', 'latitud', 'longitud']].head())
else:
    df_limpio = pd.DataFrame()
    print("⚠️ No se pudieron cargar los datos para continuar.")

In [ ]:
if not df_limpio.empty:
    def clasificar_ica(valor):
        """Clasifica el valor de PM2.5 según el estándar del ICA (EPA/Colombia)."""
        if pd.isna(valor):
            return np.nan, 'Sin dato', 'gray'
        elif valor <= 12:
            return 1, 'Buena', '#00E400'
        elif valor <= 35.4:
            return 2, 'Moderada', '#FFFF00'
        elif valor <= 55.4:
            return 3, 'Mala para grupos sensibles', '#FF7E00'
        elif valor <= 150.4:
            return 4, 'Mala', '#FF0000'
        elif valor <= 250.4:
            return 5, 'Muy mala', '#8F3F97'
        else:
            return 6, 'Peligrosa', '#7E0023'

    # Aplicar la función y expandir en 3 columnas nuevas
    df_limpio[['ica_nivel', 'ica_categoria', 'ica_color']] = df_limpio['valor_pm25'].apply(
        lambda x: pd.Series(clasificar_ica(x))
    )
    
    print("✅ Clasificación ICA aplicada correctamente.")
    print("\nDistribución de categorías:")
    display(df_limpio['ica_categoria'].value_counts())

## 4. 📊 VISUALIZACIÓN Y ANÁLISIS EXPLORATORIO

In [ ]:
if not df_limpio.empty:
    # Gráfico 1: Distribución de PM2.5 (Histograma interactivo)
    fig_hist = px.histogram(
        df_limpio, x='valor_pm25', nbins=30, 
        title='Distribución de las Concentraciones de PM2.5',
        labels={'valor_pm25': 'PM2.5 (µg/m³)', 'count': 'Frecuencia'},
        color_discrete_sequence=['#2E86AB']
    )
    fig_hist.add_vline(x=df_limpio['valor_pm25'].mean(), line_dash="dash", line_color="red",
                       annotation_text=f"Media: {df_limpio['valor_pm25'].mean():.1f}")
    fig_hist.show()
    
    # Gráfico 2: Promedio por estación (Top 8)
    top_estaciones = df_limpio.groupby('estacion')['valor_pm25'].mean().sort_values(ascending=False).head(8).reset_index()
    fig_bar = px.bar(
        top_estaciones, x='estacion', y='valor_pm25',
        title='Top 8 Estaciones con Mayor Promedio de PM2.5',
        labels={'valor_pm25': 'Promedio PM2.5 (µg/m³)', 'estacion': 'Estación'},
        color='valor_pm25',
        color_continuous_scale='YlOrRd'
    )
    fig_bar.update_layout(xaxis_tickangle=-45)
    fig_bar.show()

## 5. 🗺️ GEORREFERENCIACIÓN DE ESTACIONES

In [ ]:
if not df_limpio.empty:
    print("🗺️ Generando mapa interactivo...")
    
    # Coordenadas centrales del Valle de Aburrá
    centro_mapa = [6.2518, -75.5636]
    mapa_siata = folium.Map(location=centro_mapa, zoom_start=11, tiles='CartoDB Positron')
    cluster = MarkerCluster().add_to(mapa_siata)
    
    # Tomar la última medición de cada estación
    ultimas_mediciones = df_limpio.sort_values('fecha_local').groupby('estacion').last().reset_index()
    
    for _, fila in ultimas_mediciones.iterrows():
        html_popup = f"""
        <div style="font-family: Arial; font-size: 13px;">
            <b>📍 Estación:</b> {fila['estacion']}<br>
            <b>🌬️ PM2.5:</b> {fila['valor_pm25']:.1f} µg/m³<br>
            <b>⚠️ ICA:</b> <span style="color:{fila['ica_color']}; font-weight:bold;">{fila['ica_categoria']}</span><br>
            <b>🕒 Actualizado:</b> {fila['fecha_local'].strftime('%d/%m %H:%M')}
        </div>
        """
        
        folium.CircleMarker(
            location=[fila['latitud'], fila['longitud']],
            radius=8,
            popup=folium.Popup(html_popup, max_width=250),
            tooltip=f"{fila['estacion']}: {fila['ica_categoria']}",
            color=fila['ica_color'],
            fill=True,
            fillColor=fila['ica_color'],
            fillOpacity=0.8,
            weight=1.5
        ).add_to(cluster)
    
    display(mapa_siata)
    print(f"✅ Mapa renderizado con {len(ultimas_mediciones)} estaciones activas.")

## 6. 📈 RESUMEN ESTADÍSTICO Y CONCLUSIONES

In [ ]:
if not df_limpio.empty:
    print("="*70)
    print(" 📊 RESUMEN ESTADÍSTICO POR ESTACIÓN ".center(70))
    print("="*70)
    
    # Calcular estadísticas agrupadas
    resumen = df_limpio.groupby('estacion').agg(
        mediciones=('valor_pm25', 'count'),
        promedio=('valor_pm25', 'mean'),
        maximo=('valor_pm25', 'max'),
        minimo=('valor_pm25', 'min')
    ).round(2).sort_values(by='promedio', ascending=False)
    
    display(resumen)
    
    print("\n🏆 TOP 3 ESTACIONES CON MAYOR PROMEDIO:")
    display(resumen.head(3)[['promedio', 'maximo']])
    
    print("\n🌟 TOP 3 ESTACIONES CON MENOR PROMEDIO:")
    display(resumen.tail(3)[['promedio', 'minimo']])

In [ ]:
if not df_limpio.empty:
    print("\n" + "="*70)
    print(" 🌍 CONCLUSIÓN GENERAL DEL MONITOREO ".center(70))
    print("="*70)
    
    total_registros = len(df_limpio)
    estaciones_unicas = df_limpio['estacion'].nunique()
    
    print(f"\n📌 Se analizaron {total_registros:,} mediciones válidas provenientes de {estaciones_unicas} estaciones.")
    print(f"📌 El promedio general de PM2.5 en el período fue de {df_limpio['valor_pm25'].mean():.2f} µg/m³.")
    
    print("\n📌 Distribución porcentual de la Calidad del Aire:")
    for cat in df_limpio['ica_categoria'].unique():
        if cat != 'Sin dato':
            pct = (df_limpio['ica_categoria'] == cat).mean() * 100
            print(f"   • {cat}: {pct:.1f}%")
            
    print("\n" + "="*70)
    print(" ✅ ANÁLISIS FINALIZADO CORRECTAMENTE ".center(70))
    print("="*70)